# 1 — Préparation des supports pédagogiques

Projet TALN N11 : génération de QCM avec validation automatique. Cette étape recense les six supports fournis, extrait leur texte, applique un OCR local aux pages scannées et prépare des passages traçables par document et page.

Les PDF fournis par l’enseignant sont les sources du corpus. Les fichiers restent en local ; aucune API externe n’est appelée.

In [1]:
from pathlib import Path
import sys
import random
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'requirements.txt').exists():
    ROOT = ROOT.parent
if not (ROOT / 'requirements.txt').exists():
    raise FileNotFoundError('Racine du projet introuvable : requirements.txt est absent.')
sys.path.insert(0, str(ROOT))

from src.qcm_pipeline import extract_course_passages, split_passages

SOURCE_DIR = ROOT / 'data' / 'sources'
PROCESSED_DIR = ROOT / 'data' / 'processed'
RESULTS_DIR = ROOT / 'results'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
pdf_files = sorted(SOURCE_DIR.glob('*.pdf'))
if len(pdf_files) < 5:
    raise FileNotFoundError(f'Au moins cinq PDF attendus dans {SOURCE_DIR}; {len(pdf_files)} trouvé(s).')
print(f'Racine : {ROOT}')
print(f'Supports trouvés : {len(pdf_files)}')
pd.DataFrame({'fichier': [p.name for p in pdf_files], 'taille_Mo': [round(p.stat().st_size / 1e6, 2) for p in pdf_files]})

Racine : D:\Generateur_QCM
Supports trouvés : 6


,fichier,taille_Mo
0,01_vision_ordinateur_ch1.pdf,23.41
1,02_vision_ordinateur_ch2.pdf,17.90
2,03_employabilite_ch1.pdf,2.90
3,04_employabilite_ch2.pdf,2.51
4,05_mlops_fondamentaux.pdf,1.33
5,06_spark_dataframes_sparksql.pdf,0.73


## Extraction, OCR et segmentation

Chaque ligne de `pages` conserve la source, le numéro de page et la méthode d’extraction. L’OCR ne s’applique qu’aux pages où le texte PDF est absent ou très court. La segmentation reste volontairement simple ; les tableaux et équations peuvent nécessiter une relecture.

In [2]:
pages = extract_course_passages(SOURCE_DIR, use_ocr=True)
passages = split_passages(pages)
pages.to_csv(PROCESSED_DIR / 'pages_extraites.csv', index=False, encoding='utf-8-sig')
passages.to_csv(PROCESSED_DIR / 'passages.csv', index=False, encoding='utf-8-sig')

rapport = (pages.groupby(['source_doc', 'extraction_method'], as_index=False)
           .agg(nb_pages=('page', 'nunique'), caracteres=('text', lambda s: s.str.len().sum())))
rapport = rapport.merge(passages.groupby('source_doc').size().rename('nb_passages'), on='source_doc', how='left')
rapport['nb_passages'] = rapport['nb_passages'].fillna(0).astype(int)
rapport.to_csv(RESULTS_DIR / 'rapport_preparation.csv', index=False, encoding='utf-8-sig')
print(f'{len(pages)} pages textuelles conservées ; {len(passages)} passages utilisables.')
display(rapport)
display(passages[['source_doc', 'page', 'extraction_method', 'text']].head(8))

301 pages textuelles conservées ; 690 passages utilisables.


,source_doc,extraction_method,nb_pages,caracteres,nb_passages
0,01_vision_ordinateur_ch1.pdf,texte_insuffisant,17,487,114
1,01_vision_ordinateur_ch1.pdf,texte_pdf,67,18281,114
2,02_vision_ordinateur_ch2.pdf,texte_insuffisant,16,144,127
3,02_vision_ordinateur_ch2.pdf,texte_pdf,60,19924,127
4,03_employabilite_ch1.pdf,ocr_local,4,7866,23
5,04_employabilite_ch2.pdf,ocr_local,4,5022,16
6,05_mlops_fondamentaux.pdf,texte_pdf,28,13449,80
7,06_spark_dataframes_sparksql.pdf,texte_pdf,105,50553,330


,source_doc,page,extraction_method,text
0,01_vision_ordinateur_ch1.pdf,4,texte_pdf,L’œil humain ne perçoit pas la couleur comme u...
1,01_vision_ordinateur_ch1.pdf,4,texte_pdf,1.Le cerveau devine d’abord la couleur de la s...
2,01_vision_ordinateur_ch1.pdf,4,texte_pdf,Chaque cerveau estime la lumière différemment ...
3,01_vision_ordinateur_ch1.pdf,5,texte_pdf,La machine ne possède pas cette capacité d’ada...
4,01_vision_ordinateur_ch1.pdf,5,texte_pdf,Différence fondamentale entre vision humaine e...
5,01_vision_ordinateur_ch1.pdf,8,texte_pdf,Chaque élément de l'image est appelé « pixel »...
6,01_vision_ordinateur_ch1.pdf,9,texte_pdf,La vision par ordinateur (Computer Vision) est...
7,01_vision_ordinateur_ch1.pdf,10,texte_pdf,Tâches dans lesquelles le système traite ou an...


## Contrôles de qualité

Le corpus doit inclure au moins cinq supports. Le tableau rend visibles les pages sans texte et le recours à l’OCR. Les passages trop courts sont écartés ; les questions créées resteront à relire pour repérer les artefacts OCR.

In [3]:
assert pages['source_doc'].nunique() >= 5, 'Le corpus doit contenir au moins cinq supports.'
assert not passages.empty, 'Aucun passage exploitable : vérifier les PDF et les sorties OCR.'
assert passages['text'].str.len().min() >= 55
print('Contrôles réussis.')
print('Répartition des méthodes :')
display(pages['extraction_method'].value_counts().rename_axis('méthode').to_frame('pages'))
print('Fichiers produits :', PROCESSED_DIR / 'passages.csv', 'et', RESULTS_DIR / 'rapport_preparation.csv')

Contrôles réussis.
Répartition des méthodes :


,pages
méthode,
texte_pdf,260
texte_insuffisant,33
ocr_local,8


Fichiers produits : D:\Generateur_QCM\data\processed\passages.csv et D:\Generateur_QCM\results\rapport_preparation.csv
